## Investment return calculator

A client-friendly tool that asks simple, non-technical questions e.g (name, investment type, ticker, number of shares/bonds bought, purchase date)
and returns a plain-English summary of total returns to date, made up of:
    1. Price appreciation (capital gain/loss)
    2. Income earned (dividends for stocks/REITs, coupons for bonds)

It can also project a future value if the client wants a rough outlook, using the investment's own historical growth rate behind the scenes (the client is never asked about volatility, CAGR, beta, etc.).

In [1]:
import sys
from datetime import datetime, date
import yfinance as yf

In [2]:
def ask(prompt, cast=str, default=None):
    """Ask a question, retry on bad input, optionally allow a default."""
    while True:
        raw = input(prompt).strip()
        if not raw and default is not None:
            return default
        try:
            return cast(raw)
        except ValueError:
            print("  Sorry, I didn't get that — please try again.")

def ask_date(prompt):
    while True:
        raw = input(prompt).strip()
        for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
            try:
                return datetime.strptime(raw, fmt).date()
            except ValueError:
                continue
        print("  Please enter the date like 2023-01-15 (YYYY-MM-DD).")

def money(x):
    return f"${x:,.2f}"

In [3]:
def get_client_details():
    print("\n")
    print("  INVESTMENT RETURN CALCULATOR")
    print("\n")
    name = ask("Client name: ")
    print("\nWhat type of investment is this?")
    print("  1. Equity (shares/stock)")
    print("  2. Bond")
    print("  3. REIT (real estate investment trust)")
    choice = ask("Enter 1, 2, or 3: ", int)
    group = {1: "Equity", 2: "Bond", 3: "REIT"}.get(choice, "Equity")
    return name, group

In [4]:
# Step 2a: Equity / REIT calculation
def calculate_equity_or_reit(group):
    ticker_symbol = ask(f"\nWhat is the {group.lower()} ticker symbol (e.g. AAPL, VNQ)? ").upper()
    shares = ask("How many shares did the client purchase? ", float)
    purchase_date = ask_date("What date did the client buy in (YYYY-MM-DD)? ")

    print(f"\nLooking up {ticker_symbol}...")
    tkr = yf.Ticker(ticker_symbol)

    # Historical prices from purchase date to now
    hist = tkr.history(start=purchase_date.isoformat(), auto_adjust=False)
    if hist.empty:
        print("  Couldn't find price history for that ticker/date. Please check the symbol.")
        sys.exit(1)

    purchase_price = float(hist["Close"].iloc[0])
    current_price = float(hist["Close"].iloc[-1])

    # Dividends paid since purchase, per share, summed
    dividends = tkr.dividends
    if dividends is not None and not dividends.empty:
        dividends_since = dividends[dividends.index.date >= purchase_date]
        dividend_per_share = float(dividends_since.sum())
    else:
        dividend_per_share = 0.0

    initial_value = purchase_price * shares
    current_market_value = current_price * shares
    income_earned = dividend_per_share * shares
    price_gain = current_market_value - initial_value
    total_return_amount = price_gain + income_earned
    total_return_pct = (total_return_amount / initial_value) * 100 if initial_value else 0

    years_held = max((date.today() - purchase_date).days / 365.25, 1e-6)
    # Growth rate implied by the investment's own history — used quietly for projections only
    implied_annual_growth = (current_market_value / initial_value) ** (1 / years_held) - 1 if initial_value else 0

    return { "label": f"{ticker_symbol} ({group})", "shares_or_units": shares, "purchase_date": purchase_date, "purchase_price": purchase_price,
        "current_price": current_price, "initial_value": initial_value, "current_market_value": current_market_value, "income_earned": income_earned,
        "price_gain": price_gain, "total_return_amount": total_return_amount, "total_return_pct": total_return_pct,
        "implied_annual_growth": implied_annual_growth, }

In [5]:
# Step 2b: Bond calculation

def calculate_bond():
    face_value = ask("\nWhat is the face value of ONE bond (e.g. 1000)? ", float)
    num_bonds = ask("How many bonds did the client purchase? ", float)
    coupon_rate_pct = ask("What is the annual coupon (interest) rate, as a percent (e.g. 8)? ", float)
    purchase_price_pct = ask( "What price did the client pay per bond, as % of face value (e.g. 100 for par, 98 for a discount)? ",
        float, default=100.0 )
    purchase_date = ask_date("What date did the client buy in (YYYY-MM-DD)? ")
    current_price_pct = ask( "What is the bond's current market price, as % of face value? "
        "(if unsure/still holding to maturity, just enter the same number as purchase price): ", float, default=purchase_price_pct )

    purchase_price = face_value * (purchase_price_pct / 100)
    current_price = face_value * (current_price_pct / 100)

    initial_value = purchase_price * num_bonds
    current_market_value = current_price * num_bonds

    years_held = max((date.today() - purchase_date).days / 365.25, 0)
    annual_coupon_per_bond = face_value * (coupon_rate_pct / 100)
    income_earned = annual_coupon_per_bond * num_bonds * years_held

    price_gain = current_market_value - initial_value
    total_return_amount = price_gain + income_earned
    total_return_pct = (total_return_amount / initial_value) * 100 if initial_value else 0

    implied_annual_growth = (total_return_pct / 100) / years_held if years_held > 0 else 0

    return { "label": f"Bond (face value {money(face_value)} each)", "shares_or_units": num_bonds, "purchase_date": purchase_date,
        "purchase_price": purchase_price, "current_price": current_price, "initial_value": initial_value, "current_market_value": current_market_value,
        "income_earned": income_earned, "price_gain": price_gain, "total_return_amount": total_return_amount,  "total_return_pct": total_return_pct,
        "implied_annual_growth": implied_annual_growth,}

In [6]:
# Step 3
def print_summary(name, group, result):
    print("\n")
    print(f"  RETURN SUMMARY FOR {name.upper()}")
    print(f"Investment:            {result['label']}")
    print(f"Units held:            {result['shares_or_units']:,.2f}")
    print(f"Purchase date:         {result['purchase_date']}")
    print(f"Initial amount invested: {money(result['initial_value'])}")
    print(f"Current value:         {money(result['current_market_value'])}")
    print(f"Price gain/(loss):     {money(result['price_gain'])}")
    print(f"Income earned so far:  {money(result['income_earned'])}  "
          f"({'dividends' if group != 'Bond' else 'coupon interest'})")
    print(f"TOTAL RETURN:          {money(result['total_return_amount'])} "
          f"({result['total_return_pct']:.2f}% since purchase)")
    print("\n")

def offer_projection(result):
    answer = ask("\nWould the client like a rough projection of future value? (yes/no): ", str, default="no").lower()
    if not answer.startswith("y"):
        return
    years = ask("Project forward how many years? ", float)
    rate = result["implied_annual_growth"]
    projected_value = result["current_market_value"] * ((1 + rate) ** years)
    projected_gain = projected_value - result["current_market_value"]
    print(f"\nBased on how this investment has performed since purchase, "
          f"in {years:.0f} year(s) the estimated value could be approximately "
          f"{money(projected_value)} (an additional {money(projected_gain)} from today).")
    print("Note: this is a simple projection based on past performance and is not guaranteed.")

In [7]:
def main():
    name, group = get_client_details()

    if group == "Bond":
        result = calculate_bond()
    else:
        result = calculate_equity_or_reit(group)

    print_summary(name, group, result)
    offer_projection(result)

if __name__ == "__main__":
    main()



  INVESTMENT RETURN CALCULATOR




Client name:  Ebere



What type of investment is this?
  1. Equity (shares/stock)
  2. Bond
  3. REIT (real estate investment trust)


Enter 1, 2, or 3:  2

What is the face value of ONE bond (e.g. 1000)?  1000
How many bonds did the client purchase?  300
What is the annual coupon (interest) rate, as a percent (e.g. 8)?  7
What price did the client pay per bond, as % of face value (e.g. 100 for par, 98 for a discount)?  980
What date did the client buy in (YYYY-MM-DD)?  2023-12-11
What is the bond's current market price, as % of face value? (if unsure/still holding to maturity, just enter the same number as purchase price):  980




  RETURN SUMMARY FOR EBERE
Investment:            Bond (face value $1,000.00 each)
Units held:            300.00
Purchase date:         2023-12-11
Initial amount invested: $2,940,000.00
Current value:         $2,940,000.00
Price gain/(loss):     $0.00
Income earned so far:  $54,965.09  (coupon interest)
TOTAL RETURN:          $54,965.09 (1.87% since purchase)





Would the client like a rough projection of future value? (yes/no):  no
